# ABIDE I — Responsible AI exploratory analysis

Exploratory analysis of ABIDE I phenotype data and CPAC `filt_noglobal` Craddock CC200 ROI time series. The notebook displays and saves plots/figures in `/content/abide_eda_figures`.

**No ZIP archive is created or downloaded.** ROI time-series files are stored only in the Colab runtime so the notebook can calculate connectivity plots. The correlation matrices are exploratory results, not BrainWaveNet model outputs and do not reproduce its CWT preprocessing.

## 1. Environment and phenotype data

In [ ]:
from pathlib import Path
import os, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, chi2_contingency
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
FIGURE_DIR = Path("/content/abide_eda_figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PHENO_PATH = Path("/content/Phenotypic_V1_0b_preprocessed1.csv")
if not PHENO_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if "Phenotypic_V1_0b_preprocessed1.csv" not in uploaded:
        raise FileNotFoundError("Upload Phenotypic_V1_0b_preprocessed1.csv")
df_raw = pd.read_csv(PHENO_PATH, low_memory=False)
print("Phenotype table:", df_raw.shape)
display(df_raw.head())

## 2. Harmonise variables and inspect cohort

In [ ]:
ALIASES = {
    "subject_id": ["SUB_ID", "SUBJECT_ID", "subject", "participant_id"],
    "FILE_ID": ["FILE_ID"],
    "diagnosis": ["DX_GROUP", "DX", "diagnosis"],
    "age": ["AGE_AT_SCAN", "AGE", "age_at_scan"],
    "sex": ["SEX", "GENDER"],
    "site": ["SITE_ID", "SITE", "site_id"],
    "func_mean_fd": ["func_mean_fd", "mean_fd", "FD_mean"],
}
def find_col(names, columns):
    norm = {re.sub(r"[^a-z0-9]", "", str(c).lower()): c for c in columns}
    return next((norm[re.sub(r"[^a-z0-9]", "", a.lower())]
                 for a in names if re.sub(r"[^a-z0-9]", "", a.lower()) in norm), None)

COLUMN_MAP = {k: find_col(v, df_raw.columns) for k, v in ALIASES.items()}
display(pd.DataFrame({"analysis_variable": COLUMN_MAP.keys(),
                      "source_column": COLUMN_MAP.values()}))
if COLUMN_MAP["diagnosis"] is None:
    raise ValueError("Could not identify diagnosis column; update ALIASES.")

df = pd.DataFrame(index=df_raw.index)
for key, source in COLUMN_MAP.items():
    if source is not None:
        df[key] = df_raw[source]
for col in ["subject_id", "FILE_ID", "site"]:
    if col in df:
        df[col] = df[col].astype("string").str.strip()
for col in ["diagnosis", "age", "sex", "func_mean_fd"]:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

observed = set(df["diagnosis"].dropna().unique())
if observed.issubset({1, 2}):
    diagnosis_map = {1: "ASD", 2: "Control"}
elif observed.issubset({0, 1}):
    print("WARNING: diagnosis codes {0,1}; confirm the data dictionary.")
    diagnosis_map = {1: "ASD", 0: "Control"}
else:
    raise ValueError(f"Unrecognised diagnosis coding: {sorted(observed)}")
df["diagnosis_label"] = df["diagnosis"].map(diagnosis_map)

print("Diagnosis counts:")
display(df["diagnosis_label"].value_counts(dropna=False).to_frame("n"))
print("Age by diagnosis:")
display(df.groupby("diagnosis_label")["age"].describe())
print("Sex codes by diagnosis (ABIDE coding should be checked against data dictionary):")
display(pd.crosstab(df["sex"], df["diagnosis_label"]))

## 3. Demographic representation and site composition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(data=df, x="age", hue="diagnosis_label", bins=25,
             element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Age distribution by diagnosis")
sex_plot = df.copy()
sex_plot["sex_label"] = sex_plot["sex"].map({1: "Male (1)", 2: "Female (2)"})
sex_plot["sex_label"] = sex_plot["sex_label"].fillna(sex_plot["sex"].astype("string"))
sns.countplot(data=sex_plot, x="sex_label", hue="diagnosis_label", ax=axes[1])
axes[1].set_title("Recorded sex composition")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig_demographics.png", dpi=180, bbox_inches="tight")
plt.show()

if "site" in df:
    site_counts = df["site"].fillna("Missing").value_counts()
    display(site_counts.rename("participants").to_frame())
    site_mix = pd.crosstab(df["site"], df["diagnosis_label"], normalize="index")
    site_mix.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="Set2")
    plt.title("Diagnostic-group proportions within acquisition site")
    plt.ylabel("Within-site proportion")
    plt.xticks(rotation=65, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig_site_diagnosis_mix.png", dpi=180, bbox_inches="tight")
    plt.show()

# Missingness in original phenotype fields and key analysis variables.
missing = pd.DataFrame({
    "missing_n": df_raw.isna().sum(),
    "missing_percent": df_raw.isna().mean().mul(100)
}).sort_values("missing_percent", ascending=False)
display(missing.head(25))

## 4. Descriptive group comparisons

In [ ]:
# Mann–Whitney U for age; chi-square for sex. These are descriptive association
# tests, not measures of fairness, clinical importance or causality.
groups = {
    label: part["age"].dropna().to_numpy()
    for label, part in df.groupby("diagnosis_label")
}
if all(label in groups and len(groups[label]) for label in ["ASD", "Control"]):
    stat, p = mannwhitneyu(groups["ASD"], groups["Control"], alternative="two-sided")
    print(f"Age Mann–Whitney U={stat:.2f}, p={p:.4g}")

if df["sex"].notna().any():
    contingency = pd.crosstab(df["sex"], df["diagnosis_label"])
    display(contingency)
    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        chi2, p, dof, expected = chi2_contingency(contingency)
        print(f"Sex-by-diagnosis chi-square={chi2:.2f}, df={dof}, p={p:.4g}")

## 5. Download CC200 ROI time series (for connectivity figures)

This cell downloads the CPAC `filt_noglobal` `rois_cc200` derivatives directly into the Colab runtime. No ZIP is created and nothing is downloaded to your computer. Files are kept in `/content/ABIDE_ROI_CC200/data` for the following connectivity analysis.

In [ ]:
import requests

ROI_ROOT = Path("/content/ABIDE_ROI_CC200")
ROI_DIR = ROI_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = ("https://s3.amazonaws.com/fcp-indi/data/Projects/"
            "ABIDE_Initiative/Outputs/cpac/filt_noglobal/rois_cc200")
MAX_WORKERS, MAX_RETRIES, EXPECTED_ROIS = 5, 3, 200

if "FILE_ID" not in df_raw:
    raise KeyError("FILE_ID is required to download ROI derivatives.")
file_ids = sorted(set(
    df_raw["FILE_ID"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s.str.lower() != "no_filename")]
))
print("Unique usable FILE_IDs:", len(file_ids))

def valid_roi(path):
    try:
        x = np.loadtxt(path)
        return x.ndim == 2 and x.shape[0] >= 3 and x.shape[1] == 200 and np.isfinite(x).all()
    except Exception:
        return False

def download_roi(file_id):
    name = f"{file_id}_rois_cc200.1D"
    dest = ROI_DIR / name
    if dest.exists() and valid_roi(dest):
        return {"FILE_ID": file_id, "status": "already_downloaded", "error": ""}
    url = f"{BASE_URL}/{name}"
    tmp = dest.with_suffix(".1D.tmp")
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, timeout=(20, 120))
            if response.status_code == 404:
                return {"FILE_ID": file_id, "status": "not_found", "error": "HTTP 404"}
            response.raise_for_status()
            tmp.write_bytes(response.content)
            if not valid_roi(tmp):
                raise ValueError("Invalid file: expected finite matrix with 200 columns")
            os.replace(tmp, dest)
            return {"FILE_ID": file_id, "status": "downloaded", "error": ""}
        except Exception as exc:
            if tmp.exists():
                tmp.unlink()
            if attempt == MAX_RETRIES:
                return {"FILE_ID": file_id, "status": "failed", "error": str(exc)}
            time.sleep(2 * attempt)

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(download_roi, fid) for fid in file_ids]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading ROI"):
        results.append(future.result())

manifest = pd.DataFrame(results).sort_values("FILE_ID")
display(manifest.status.value_counts().to_frame("participants"))

dims = []
for fp in sorted(ROI_DIR.glob("*.1D")):
    if valid_roi(fp):
        x = np.loadtxt(fp)
        dims.append({"FILE_ID": fp.name.removesuffix("_rois_cc200.1D"),
                     "timepoints": x.shape[0], "n_roi": x.shape[1]})
dimensions = pd.DataFrame(dims)
display(dimensions[["timepoints", "n_roi"]].describe())


print("ROI files ready for plotting:", len(dimensions))

## 6. Compute participant-level functional connectivity

The filename's `FILE_ID` is matched exactly to the phenotype table. Each output matrix remains 200 × 200. Constant or insufficiently observed ROI columns are left as `NaN` rows/columns and counted; undefined correlations are never silently replaced by zero.

In [ ]:
ROI_DIR = Path("/content/ABIDE_ROI_CC200/data")
if not ROI_DIR.exists():
    raise FileNotFoundError("Run the ROI download cell first or set ROI_DIR to your data folder.")

phenotype = df.copy()
phenotype["FILE_ID"] = phenotype["FILE_ID"].astype("string").str.strip()
phenotype = phenotype.dropna(subset=["FILE_ID"])
phenotype = phenotype[
    (phenotype.FILE_ID != "") & (phenotype.FILE_ID.str.lower() != "no_filename")
]
duplicate_ids = phenotype[phenotype.FILE_ID.duplicated(keep=False)]
print("Duplicate phenotype FILE_ID rows:", len(duplicate_ids))
phenotype_lookup = phenotype.drop_duplicates("FILE_ID").set_index("FILE_ID")

connectivity, records, skipped = {}, [], []
for fp in sorted(ROI_DIR.rglob("*.1D")):
    file_id = fp.name.removesuffix("_rois_cc200.1D")
    try:
        x = np.loadtxt(fp, dtype=float)
        if x.ndim != 2 or x.shape[0] < 3 or x.shape[1] != 200:
            raise ValueError(f"Unexpected time-series shape: {x.shape}")

        valid_roi = np.zeros(x.shape[1], dtype=bool)
        for j in range(x.shape[1]):
            values = x[np.isfinite(x[:, j]), j]
            valid_roi[j] = len(values) >= 3 and np.std(values) > 1e-12
        corr = pd.DataFrame(x).corr(min_periods=3).to_numpy()
        corr[~valid_roi, :] = np.nan
        corr[:, ~valid_roi] = np.nan
        upper = corr[np.triu_indices(200, k=1)]
        edges = upper[np.isfinite(upper)]
        if len(edges) == 0:
            raise ValueError("No valid off-diagonal correlations")

        connectivity[file_id] = corr
        row = phenotype_lookup.loc[file_id] if file_id in phenotype_lookup.index else None
        records.append({
            "FILE_ID": file_id,
            "subject_id": row.get("subject_id", np.nan) if row is not None else np.nan,
            "diagnosis_label": row.get("diagnosis_label", np.nan) if row is not None else np.nan,
            "site": row.get("site", np.nan) if row is not None else np.nan,
            "timepoints": x.shape[0],
            "n_roi_invalid": int((~valid_roi).sum()),
            "n_valid_edges": len(edges),
            "mean_abs_connectivity": float(np.mean(np.abs(edges))),
            "phenotype_matched": row is not None
        })
    except Exception as exc:
        skipped.append({"filename": fp.name, "FILE_ID": file_id, "reason": str(exc)})

conn_df = pd.DataFrame(records)
skipped_df = pd.DataFrame(skipped)
print("ROI files:", len(list(ROI_DIR.rglob("*.1D"))))
print("Matrices computed:", len(connectivity))
print("Skipped files:", len(skipped_df))
print("Phenotype matched:", int(conn_df.phenotype_matched.sum()) if not conn_df.empty else 0)
if not conn_df.empty:
    display(conn_df[["timepoints", "n_roi_invalid", "n_valid_edges",
                     "mean_abs_connectivity", "phenotype_matched"]].describe())
else:
    print("No valid matrices. Inspect ROI filenames, file format and skipped files.")

## 7. Connectivity results and matching diagnostics

In [ ]:
if conn_df.empty:
    raise ValueError("No valid connectivity matrices are available; inspect the skipped-files CSV.")

roi_ids = set(conn_df.FILE_ID)
phenotype_ids = set(phenotype_lookup.index)
unmatched_roi = sorted(roi_ids - phenotype_ids)
phenotype_without_roi = sorted(phenotype_ids - roi_ids)
print("ROI IDs without phenotype match:", len(unmatched_roi))
print("Phenotype IDs without valid ROI:", len(phenotype_without_roi))



matched = conn_df[conn_df.phenotype_matched & conn_df.diagnosis_label.isin(["ASD", "Control"])].copy()
if not matched.empty:
    example_id = next(iter(connectivity))
    sns.heatmap(connectivity[example_id], cmap="vlag", center=0, vmin=-1, vmax=1,
                square=True, xticklabels=False, yticklabels=False)
    plt.title(f"Example CC200 connectivity matrix: {example_id}")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig_connectivity_example.png", dpi=180, bbox_inches="tight")
    plt.show()

    display(matched.groupby("diagnosis_label")["mean_abs_connectivity"]
            .agg(["count", "mean", "std", "median"]))
    sns.boxplot(data=matched, x="diagnosis_label", y="mean_abs_connectivity",
                order=["ASD", "Control"])
    plt.title("Mean absolute ROI connectivity by diagnostic group")
    plt.ylabel("Mean |Pearson r| across valid ROI pairs")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig_connectivity_by_group.png", dpi=180, bbox_inches="tight")
    plt.show()

    a = matched.loc[matched.diagnosis_label == "ASD", "mean_abs_connectivity"].dropna().to_numpy()
    c = matched.loc[matched.diagnosis_label == "Control", "mean_abs_connectivity"].dropna().to_numpy()
    if len(a) > 1 and len(c) > 1:
        pooled = np.sqrt(((len(a)-1)*np.var(a, ddof=1) + (len(c)-1)*np.var(c, ddof=1))
                         / (len(a)+len(c)-2))
        if pooled > 0:
            d = (a.mean() - c.mean()) / pooled
            hedges_g = (1 - 3/(4*(len(a)+len(c))-9)) * d
            print(f"Hedges' g (ASD minus Control): {hedges_g:.3f}")
            print("Descriptive effect size only; not a causal or fairness measure.")

    if "site" in matched:
        site_summary = matched.groupby(["site", "diagnosis_label"])["mean_abs_connectivity"].agg(
            ["count", "mean", "std"]).reset_index()
        display(site_summary)


## 7.2. Motion sensitivity and site-adjusted connectivity analysis

This section links participant-level mean absolute connectivity to the phenotype table using the exact FILE_ID. It examines motion (func_mean_fd), compares groups with a Welch confidence interval, and estimates the ASD–Control association after adjusting for age, sex, mean framewise displacement and site. These are observational sensitivity analyses, not causal estimates or evidence of model fairness.

In [ ]:
# Merge motion and covariates from the harmonised phenotype table.
required = ["FILE_ID", "diagnosis_label", "age", "sex", "site", "func_mean_fd"]
lookup = phenotype_lookup.reset_index()
missing = [c for c in required if c not in lookup.columns]
if missing:
    raise ValueError("Required phenotype variables are unavailable: " + ", ".join(missing)
                     + ". Check the ABIDE CSV column names / ALIASES.")

analysis = matched.merge(lookup[required], on="FILE_ID", how="left",
                         suffixes=("", "_phenotype"))
for col in ["diagnosis_label", "age", "sex", "site", "func_mean_fd"]:
    phenotype_col = f"{col}_phenotype"
    if phenotype_col in analysis:
        analysis[col] = analysis[phenotype_col]
analysis["func_mean_fd"] = pd.to_numeric(analysis["func_mean_fd"], errors="coerce")
analysis["age"] = pd.to_numeric(analysis["age"], errors="coerce")
analysis["sex"] = analysis["sex"].astype("string").str.strip()
analysis = analysis.replace([np.inf, -np.inf], np.nan)

print("Connectivity participants:", len(analysis))
print("Participants with mean FD:", int(analysis["func_mean_fd"].notna().sum()))
model_cols = ["mean_abs_connectivity", "diagnosis_label", "age", "sex", "site", "func_mean_fd"]
print("Participants complete for adjusted model:",
      int(analysis[model_cols].dropna().shape[0]))

# 1) Motion distribution and group comparison.
motion = analysis.dropna(subset=["func_mean_fd", "diagnosis_label"]).copy()
display(motion.groupby("diagnosis_label")["func_mean_fd"].agg(
    ["count", "mean", "std", "median", "min", "max"]))
if {"ASD", "Control"}.issubset(set(motion["diagnosis_label"].unique())):
    fd_asd = motion.loc[motion.diagnosis_label == "ASD", "func_mean_fd"].to_numpy()
    fd_ctl = motion.loc[motion.diagnosis_label == "Control", "func_mean_fd"].to_numpy()
    u_fd, p_fd = mannwhitneyu(fd_asd, fd_ctl, alternative="two-sided")
    print(f"Mean FD group comparison: Mann–Whitney U={u_fd:.1f}, p={p_fd:.4g}")

plt.figure(figsize=(7, 4.5))
sns.boxplot(data=motion, x="diagnosis_label", y="func_mean_fd", order=["ASD", "Control"])
plt.title("Mean framewise displacement by diagnostic group")
plt.xlabel("Diagnostic group")
plt.ylabel("Mean framewise displacement (mm)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig_mean_fd_by_group.png", dpi=180, bbox_inches="tight")
plt.show()

# 2) Association between motion and absolute connectivity.
from scipy.stats import pearsonr, spearmanr, t as student_t, ttest_ind
motion_conn = analysis[["func_mean_fd", "mean_abs_connectivity"]].dropna()
if len(motion_conn) >= 3:
    pr, pp = pearsonr(motion_conn.func_mean_fd, motion_conn.mean_abs_connectivity)
    sr, sp = spearmanr(motion_conn.func_mean_fd, motion_conn.mean_abs_connectivity)
    print(f"Motion vs mean |r|: Pearson r={pr:.3f}, p={pp:.4g}; "
          f"Spearman rho={sr:.3f}, p={sp:.4g}")
    plt.figure(figsize=(7, 4.5))
    sns.regplot(data=motion_conn, x="func_mean_fd", y="mean_abs_connectivity",
                scatter_kws={"alpha": 0.35, "s": 20}, line_kws={"linewidth": 2})
    plt.title("Head motion and mean absolute connectivity")
    plt.xlabel("Mean framewise displacement (mm)")
    plt.ylabel("Mean absolute connectivity")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig_motion_vs_connectivity.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Insufficient complete observations to assess motion-connectivity association.")

# 3) Unadjusted ASD-Control difference with Welch 95% confidence interval.
group_data = {
    label: analysis.loc[analysis.diagnosis_label == label, "mean_abs_connectivity"].dropna().to_numpy()
    for label in ["ASD", "Control"]
}
if all(len(group_data[g]) > 1 for g in group_data):
    a, c = group_data["ASD"], group_data["Control"]
    diff = a.mean() - c.mean()
    va, vc = a.var(ddof=1), c.var(ddof=1)
    se = np.sqrt(va/len(a) + vc/len(c))
    df_welch = (va/len(a) + vc/len(c))**2 / (
        (va/len(a))**2/(len(a)-1) + (vc/len(c))**2/(len(c)-1))
    critical = student_t.ppf(0.975, df_welch)
    ci = (diff - critical*se, diff + critical*se)
    welch = ttest_ind(a, c, equal_var=False)
    print(f"ASD - Control mean difference = {diff:.4f}; "
          f"95% Welch CI [{ci[0]:.4f}, {ci[1]:.4f}], "
          f"Welch t-test p={welch.pvalue:.4g}")

# 4) Site-adjusted OLS sensitivity model with HC3 robust standard errors.
model_data = analysis[model_cols].dropna().copy()
if len(model_data) > 20 and model_data["diagnosis_label"].nunique() == 2:
    model_data["ASD_indicator"] = (model_data["diagnosis_label"] == "ASD").astype(float)
    categorical = pd.get_dummies(model_data[["sex", "site"]].astype("category"),
                                 drop_first=True, dtype=float)
    X_df = pd.concat([
        pd.Series(1.0, index=model_data.index, name="intercept"),
        model_data[["ASD_indicator", "age", "func_mean_fd"]].astype(float),
        categorical
    ], axis=1)
    X, y = X_df.to_numpy(dtype=float), model_data["mean_abs_connectivity"].to_numpy(dtype=float)
    if np.linalg.matrix_rank(X) < X.shape[1]:
        print("Adjusted model not estimated: design matrix is rank-deficient.")
    elif len(y) <= X.shape[1] + 5:
        print("Adjusted model not estimated: too few residual degrees of freedom.")
    else:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        residual = y - X @ beta
        bread = np.linalg.pinv(X.T @ X)
        leverage = np.sum((X @ bread) * X, axis=1)
        scale = (residual / np.maximum(1 - leverage, 1e-8))**2
        meat = X.T @ (X * scale[:, None])
        cov_hc3 = bread @ meat @ bread
        se_hc3 = np.sqrt(np.maximum(np.diag(cov_hc3), 0))
        residual_df = len(y) - X.shape[1]
        t_values = beta / se_hc3
        p_values = 2 * student_t.sf(np.abs(t_values), df=residual_df)
        critical = student_t.ppf(0.975, df=residual_df)
        results = pd.DataFrame({
            "term": X_df.columns, "coefficient": beta, "HC3_SE": se_hc3,
            "p_value": p_values, "CI95_low": beta-critical*se_hc3,
            "CI95_high": beta+critical*se_hc3
        })
        display(results[results.term.isin(["ASD_indicator", "age", "func_mean_fd"])].round(5))
        print(f"HC3 OLS complete-case n={len(y)}; sex and site included as fixed effects.")
        print("ASD_indicator is the adjusted ASD-minus-Control association; it is not causal.")
else:
    print("Adjusted model not estimated: insufficient complete observations or groups.")


### 7.1. Distribution of correlations in the example matrix

The diagonal is excluded because each ROI is perfectly correlated with itself (`r = 1`). The histogram and threshold proportions describe only the selected participant's matrix; they should not be generalised to the full cohort.

In [ ]:
if 'example_id' not in globals():
    example_id = next(iter(connectivity))

example_corr = connectivity[example_id]
upper = example_corr[np.triu_indices_from(example_corr, k=1)]
upper = upper[np.isfinite(upper)]
if upper.size == 0:
    raise ValueError('The example matrix has no valid off-diagonal correlations.')

summary = pd.DataFrame({
    'metric': ['Valid ROI pairs', 'Mean', 'Median', 'Standard deviation',
               'Minimum', 'Maximum',
               'Proportion r > 0.3 (%)', 'Proportion r > 0.5 (%)',
               'Proportion r > 0.7 (%)', 'Proportion r > 0.9 (%)'],
    'value': [len(upper), np.mean(upper), np.median(upper), np.std(upper),
              np.min(upper), np.max(upper),
              *(np.mean(upper > t) * 100 for t in [0.3, 0.5, 0.7, 0.9])]
})
display(summary.round(3))

plt.figure(figsize=(8, 4.5))
plt.hist(upper, bins=50, edgecolor='black')
plt.axvline(np.mean(upper), linestyle='--', label=f'Mean = {np.mean(upper):.2f}')
plt.axvline(np.median(upper), linestyle=':', label=f'Median = {np.median(upper):.2f}')
plt.xlabel('Pearson correlation (r)')
plt.ylabel('Number of ROI pairs')
plt.title(f'Off-diagonal correlation distribution: {example_id}')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fig_connectivity_distribution_example.png', dpi=180, bbox_inches='tight')
plt.show()

print('Interpretation: inspect the percentages above the thresholds alongside the histogram. Positive correlations are not, by themselves, evidence of artefact or direct anatomical connections.')

## 9. Interpretation and limitations

- Demographic imbalance and site-level differences are signals for stratified evaluation; they do not by themselves prove discriminatory model behaviour.
- P-values are descriptive and do not quantify clinical importance or fairness.
- The connectivity summary is the mean absolute off-diagonal Pearson correlation over valid ROI pairs. It is sensitive to preprocessing, motion, site effects and missing/constant ROI signals.
- Motion analyses use func_mean_fd and therefore include only participants with that QC field available. Missingness and complete-case selection must be reported.
- The adjusted OLS model includes diagnosis, age, sex, mean framewise displacement and site fixed effects, with HC3 robust standard errors. It remains observational and cannot establish causality.
- The notebook does not train BrainWaveNet and does not perform its CWT representation. The model's published preprocessing and the exact sample selection must be independently reproduced before claiming methodological equivalence.
- Figures are saved in /content/abide_eda_figures; ROI files remain in the temporary Colab runtime and are not packaged into an archive.